> This notebooks is entirely about entity matching part of  Web-Data Integration Pipeline. It's more of a exploration rather than integration for entity-matching.

> We'll experiment and ultimately decide on which methods to implement in the final pipeline in this notebook.

In [1]:
from pathlib import Path

ROOT = Path.cwd()

DATA_DIR = ROOT / "parquet"
OUTPUT_DIR = ROOT / "output"
MLDS_DIR = ROOT / "ml-datasets"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:
from PyDI.io import load_parquet

amazon_books = load_parquet(
    DATA_DIR / "amazon_new.parquet",
    name="amazon_books"
)

goodreads = load_parquet(
    DATA_DIR / "goodreads_new.parquet",
    name="goodreads"
)

metabooks = load_parquet(
  DATA_DIR / "metabooks_new.parquet",
  name="metabooks"
)

In [3]:
import logging

LOGS = OUTPUT_DIR / "logs"
LOGS.mkdir(parents=True, exist_ok=True)

logging.basicConfig(
    level=logging.INFO,
    format='[%(levelname)-5s] %(name)s - %(message)s',
    handlers=[
          logging.FileHandler(LOGS / 'pydi.log'),
          logging.StreamHandler()
      ],
    force=True
)

## Basic Dataset Summary

In [4]:
datasets = [amazon_books, goodreads, metabooks]
names = ["Amazon", "Goodreads", "Metabooks"]

for df, name in zip(datasets, names):
    print(f"{name}:")
    print(f"  Records: {len(df):,}")
    print(f"  Attributes: {len(df.columns)}")
    print(f"  Columns: {list(df.columns)}")
    print(f"  Dataset name: {df.attrs.get('dataset_name', 'unknown')}")
    print()

total_records = sum(len(df) for df in datasets)
print(f"Total records across all datasets: {total_records:,}")

Amazon:
  Records: 271,360
  Attributes: 8
  Columns: ['id', 'title', 'author', 'publish_year', 'publisher', 'isbn_clean', 'title_norm', 'author_norm']
  Dataset name: amazon_books

Goodreads:
  Records: 52,478
  Attributes: 16
  Columns: ['id', 'title', 'author', 'rating', 'rating_number', 'language', 'genres', 'book_format', 'edition', 'page_count', 'publisher', 'publish_year', 'price', 'isbn_clean', 'title_norm', 'author_norm']
  Dataset name: goodreads

Metabooks:
  Records: 535,159
  Attributes: 14
  Columns: ['id', 'title', 'author', 'rating', 'rating_number', 'language', 'genres', 'publisher', 'page_count', 'price', 'publish_year', 'isbn_clean', 'title_norm', 'author_norm']
  Dataset name: metabooks

Total records across all datasets: 858,997


In [5]:
random_common_book=list(set(amazon_books.isbn_clean)&set(goodreads.isbn_clean)&set(metabooks.isbn_clean))[60]
display(amazon_books[amazon_books.isbn_clean==random_common_book])
display(goodreads[goodreads.isbn_clean==random_common_book])
display(metabooks[metabooks.isbn_clean==random_common_book])

,id,title,author,publish_year,publisher,isbn_clean,title_norm,author_norm
248731,amazon_248732,The Namesake : A Novel,Jhumpa Lahiri,2004,Mariner Books,0618485228,the namesake a novel,jhumpa lahiri


,id,title,author,rating,rating_number,language,genres,book_format,edition,page_count,publisher,publish_year,price,isbn_clean,title_norm,author_norm
745,goodreads_746,The Namesake,Jhumpa Lahiri,4.0,233553,English,"['Fiction', 'India', 'Contemporary', 'Literary...",Paperback,nan,291,Mariner Books,2004,3.19,0618485228,the namesake,jhumpa lahiri


,id,title,author,rating,rating_number,language,genres,publisher,page_count,price,publish_year,isbn_clean,title_norm,author_norm
321156,metabooks_321157,"The Namesake By Lahiri, Jhumpa",Jhumpa Lahiri,4.3,9872,English,<NA>,Mariner / Houghton Mifflin,291,15.26,2003,0618485228,the namesake by lahiri jhumpa,jhumpa lahiri


# Entity Matching

## Blocking

> In this part we are diving deep into blocking methods to see which one performs the best. Then we are gonna pick that one for rule-based matching.

In [6]:
from PyDI.entitymatching import StandardBlocker
from PyDI.entitymatching import StringComparator, NumericComparator
from PyDI.entitymatching import RuleBasedMatcher
from PyDI.entitymatching import SortedNeighbourhoodBlocker
from PyDI.entitymatching import TokenBlocker
from PyDI.entitymatching import EmbeddingBlocker
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', 200)

from PyDI.entitymatching import EntityMatchingEvaluator
from PyDI.io import load_csv

### Metabooks - Amazon

In [7]:
BLOCK_EVAL_DIR = OUTPUT_DIR / "blocking_evaluation"
CORR_DIR = OUTPUT_DIR / "correspondences"

BLOCK_EVAL_DIR.mkdir(parents=True, exist_ok=True)
CORR_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
st_blocker_m2a = StandardBlocker(
    metabooks, amazon_books,
    on=['title_norm','author_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

standard_candidates_m2a = st_blocker_m2a.materialize()


st_blocker_m2a_norm = StandardBlocker(
    metabooks, amazon_books,
    on=['title_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)

standard_candidates_m2a_norm = st_blocker_m2a_norm.materialize()

sn_blocker_m2a_20 = SortedNeighbourhoodBlocker(
    metabooks, amazon_books,
    key='title_norm',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2a = sn_blocker_m2a_20.materialize()

sn_blocker_m2a_25 = SortedNeighbourhoodBlocker(
    metabooks, amazon_books,
    key='title_norm',
    window=25,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2a_25 = sn_blocker_m2a_25.materialize()

token_blocker_m2a = TokenBlocker(
    metabooks, amazon_books,
    column='title_norm',
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id',
    ngram_size=3,
    ngram_type='character'
)
#token_candidates_m2a = token_blocker_m2a.materialize()



embedding_blocker_m2a = EmbeddingBlocker(
    metabooks, amazon_books,
    text_cols=['title_norm', 'author_norm'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
#embedding_candidates_m2a = embedding_blocker_m2a.materialize()

standard_candidates_m2a.to_csv(CORR_DIR / "StandardBlocker-Corr-MA.csv", index=False)
sn_candidates_m2a.to_csv(CORR_DIR / "SNBlocker-Corr-MA.csv", index=False)
#token_candidates_m2a.to_csv(CORR_DIR / "TokenBlocker-Corr-MG.csv", index=False)
#embedding_candidates_m2a.to_csv(CORR_DIR / "EmbeddingBlocker-Corr-MA.csv", index=False)




[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 518708 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 248239 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 20475 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 507818 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 236280 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 31220 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurc

In [10]:
evaluator = EntityMatchingEvaluator()

# Create dictionaries of candidates for both dataset combinations
m2a_blocking_candidates = {
    'StandardBlocking': [standard_candidates_m2a, st_blocker_m2a],
    'StandardBlockingNorm': [standard_candidates_m2a_norm, st_blocker_m2a_norm],
    'SortedNeighbourhoodBlocker': [sn_candidates_m2a, sn_blocker_m2a_20],
    'SortedNeighbourhoodBlocker25': [sn_candidates_m2a, sn_blocker_m2a_25],
    #'TokenBlocking': [token_candidates_m2a,token_blocker_m2a],
    #'EmbeddingBlocking': [embedding_candidates_m2a, embedding_blocker_m2a]
}

m2a_correspondences = load_csv(
    MLDS_DIR/"train_MA.csv",
    name="m2a_test", header=1, names=['id1', 'id2', 'label'], add_index=False
)

m2a_results = []
for method_name, candidates in m2a_blocking_candidates.items():
    result = evaluator.evaluate_blocking(candidates[0], m2a_correspondences,candidates[1], out_dir="output/blocking-evaluation")
    result['method'] = method_name
    result['dataset'] = 'm2a'
    m2a_results.append(result)

m2a_best = max(m2a_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))
print(f"Best blocking for a2g: {m2a_best['method']} (PC: {m2a_best['pair_completeness']:.3f}, RR: {m2a_best['reduction_ratio']:.3f})")

[INFO ] root -   Pair Completeness: 0.415
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 51/123
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.585
[INFO ] root -   Pair Quality:      0.001
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 72/123
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.935
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 115/123
[INFO ] root - Blocking evaluation complete!
[INFO ] root -   Pair Completeness: 0.935
[INFO ] root -   Pair Quality:      0.000
[INFO ] root -   Reduction Ratio:   1.000
[INFO ] root -   True Matches Found: 115/123
[INFO ] root - Blocking evaluation complete!


Best blocking for a2g: SortedNeighbourhoodBlocker (PC: 0.935, RR: 1.000)


In [11]:
import pandas as pd

m2a_summary = pd.DataFrame(
    [
        {
            "method": res["method"],
            "pair_completeness": res["pair_completeness"],
            "reduction_ratio": res["reduction_ratio"],
        }
        for res in m2a_results 
    ]
)
m2a_summary

,method,pair_completeness,reduction_ratio
0,StandardBlocking,0.414634,1.000000
1,StandardBlockingNorm,0.585366,0.999999
2,SortedNeighbourhoodBlocker,0.934959,0.999942
3,SortedNeighbourhoodBlocker25,0.934959,0.999942


### Metabooks - Goodreads Pair

In [ ]:
st_blocker_m2g = StandardBlocker(
    metabooks, goodreads,
    on=['title','author'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
standard_candidates_m2g = st_blocker_m2g.materialize()

st_blocker_m2g_norm = StandardBlocker(
    metabooks, goodreads,
    on=['title_norm','author_norm'],
    batch_size=1000,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
standard_candidates_m2g_norm = st_blocker_m2g_norm.materialize()

sn_blocker_m2g_20 = SortedNeighbourhoodBlocker(
    metabooks, goodreads,
    key='title_norm',
    window=20,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2g = sn_blocker_m2g_20.materialize()

sn_blocker_m2g_25 = SortedNeighbourhoodBlocker(
    metabooks, goodreads,
    key='title_norm',
    window=25,
    batch_size=500,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
sn_candidates_m2g_25 = sn_blocker_m2g_25.materialize()

token_blocker_m2g = TokenBlocker(
    metabooks, goodreads,
    column='title_norm',
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id',
    ngram_size=2,
    ngram_type='character'
)
#token_candidates_m2g = token_blocker_m2g.materialize()


embedding_blocker_m2g = EmbeddingBlocker(
    metabooks, goodreads,
    text_cols=['title_norm', 'author_norm'],
    model="sentence-transformers/all-MiniLM-L6-v2",
    index_backend="sklearn",
    top_k=20,
    batch_size=200,
    output_dir=BLOCK_EVAL_DIR,
    id_column='id'
)
#embedding_candidates_m2g = embedding_blocker_m2g.materialize()

standard_candidates_m2g.to_csv(CORR_DIR / "StandardBlocker-Corr-MG.csv", index=False)
sn_candidates_m2g.to_csv(CORR_DIR / "SNBlocker-Corr-MG.csv", index=False)
#token_candidates_m2g.to_csv(CORR_DIR / "TokenBlocker-Corr-MG.csv", index=False)
#embedding_candidates_m2g.to_csv(CORR_DIR / "EmbeddingBlocker-Corr-MG.csv", index=False)

[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 518708 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 52384 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4112 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurcanmemis/Desktop/data-integration-team-project/books-integration/output/blocking_evaluation/debugResultsBlocking_StandardBlocker.csv
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 517669 blocking keys for first dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 51247 blocking keys for second dataset
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - created 4262 blocks from blocking keys
[INFO ] PyDI.entitymatching.blocking.standard.StandardBlocker - Debug results written to file: /Users/onurcanme

In [ ]:
evaluator = EntityMatchingEvaluator()

m2g_blocking_candidates = {
    'StandardBlocking': [standard_candidates_m2a, st_blocker_m2a],
    'StandardBlockingNorm': [standard_candidates_m2a_norm, st_blocker_m2a_norm],
    'SortedNeighbourhoodBlocker': [sn_candidates_m2a, sn_blocker_m2a_20],
    'SortedNeighbourhoodBlocker25': [sn_candidates_m2a, sn_blocker_m2a_25],
    #'TokenBlocking': [token_candidates_m2a,token_blocker_m2a],
    #'EmbeddingBlocking': [embedding_candidates_m2a, embedding_blocker_m2a]
}

m2g_correspondences = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="m2g_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)

m2g_results = []
for method_name, (cand_df, blocker) in m2g_blocking_candidates.items():
    result = evaluator.evaluate_blocking(
        cand_df,
        m2g_correspondences,
        blocker,
        out_dir="output/blocking-evaluation"
    )
    result['method'] = method_name
    result['dataset'] = 'm2g'
    m2g_results.append(result)

m2g_best = max(m2g_results, key=lambda x: (x['pair_completeness'], x['reduction_ratio']))
print(f"Best blocking for m2g: {m2g_best['method']} (PC: {m2g_best['pair_completeness']:.3f}, RR: {m2g_best['reduction_ratio']:.3f})")


In [ ]:
m2g_summary = pd.DataFrame(
    [
        {
            "method": res["method"],
            "pair_completeness": res["pair_completeness"],
            "reduction_ratio": res["reduction_ratio"],
        }
        for res in m2g_results 
    ]
)
m2g_summary

## Rule-Based Entity Matching

In [ ]:
from PyDI.entitymatching import RuleBasedMatcher, EntityMatchingEvaluator
from PyDI.io import load_csv
from pathlib import Path

matcher = RuleBasedMatcher()
OUTPUT_DIR = Path("output/entity-matching")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def run_matching(combo_key, left_df, right_df, candidates, comparators, weights,
                 threshold, test_file, id_column="id"):
    correspondences = matcher.match(
        df_left=left_df,
        df_right=right_df,
        candidates=candidates,
        comparators=comparators,
        weights=weights,
        threshold=threshold,
        id_column=id_column,
    )

    test_pairs = load_csv(
        MLDS_DIR / test_file,
        name=f"test_{combo_key.lower()}",
        header=1,
        names=["id1", "id2", "label"],
        add_index=False,
    )

    out_dir = OUTPUT_DIR / combo_key.lower()
    out_dir.mkdir(parents=True, exist_ok=True)

    results = EntityMatchingEvaluator.evaluate_matching(
        correspondences=correspondences,
        test_pairs=test_pairs,
        out_dir=out_dir,
    )
    results["combo"] = combo_key
    return results

# Base comparators used by both combos
comparators_base = [
    StringComparator(column="title_norm", similarity_function="cosine"),
    StringComparator(column="author_norm", similarity_function="cosine", preprocess=str.lower),
    NumericComparator(column="publish_year", max_difference=1),
]

comparators_ma = comparators_base
comparators_mg = comparators_base + [
    StringComparator(column="genres", similarity_function="jaccard",
                     preprocess=str.lower, list_strategy="concatenate"),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
]

results_ma = run_matching(
    combo_key="MA",
    left_df=metabooks,
    right_df=amazon_books,
    candidates=sn_blocker_m2a_20,
    comparators=comparators_ma,
    weights=[0.7, 0.2, 0.1],
    threshold=0.7,
    test_file="test_MA.csv",
)

results_mg = run_matching(
    combo_key="MG",
    left_df=metabooks,
    right_df=goodreads,
    candidates=sn_blocker_m2g_20,
    comparators=comparators_mg,
    weights=[0.6, 0.2, 0.05, 0.05, 0.05, 0.05],
    threshold=0.7,
    test_file="test_MG.csv",
)

display(pd.concat([results_ma, results_mg], ignore_index=True))


## ML- Based Entity Matching

In [47]:
from PyDI.io import load_parquet
from PyDI.entitymatching import FeatureExtractor
from PyDI.entitymatching import StringComparator, NumericComparator


m2g_correspondences = load_csv(
    MLDS_DIR / "train_MG.csv",
    name="m2g_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)
m2a_correspondences = load_csv(
    MLDS_DIR / "train_MA.csv",
    name="m2g_test",
    header=1,
    names=['id1', 'id2', 'label'],
    add_index=False
)

comparators_m2a = [
    # Title similarity
    StringComparator(column='title_norm',similarity_function='jaccard'),
    StringComparator(column='title_norm',similarity_function='cosine'),
    StringComparator(column='title_norm',similarity_function='jaro'),
    StringComparator(column='title_norm',similarity_function='jaro_winkler'),
    
    # author similarity
    StringComparator(column='author_norm',similarity_function='jaccard', preprocess=str.lower),
    StringComparator(column='author_norm',similarity_function='cosine', preprocess=str.lower),
    StringComparator(column='author_norm',similarity_function='jaro', preprocess=str.lower),
    # publish year similarity
    NumericComparator(column='publish_year',max_difference=1)
    ]
comparators_m2g = comparators_m2a + [
    StringComparator(column='genres',similarity_function='jaccard',preprocess=str.lower,list_strategy='concatenate'),
    NumericComparator(column="price", max_difference=5),
    NumericComparator(column="page_count", max_difference=10),
]

feature_extractor_m2a = FeatureExtractor(comparators_m2a)
feature_extractor_m2g = FeatureExtractor(comparators_m2g)

# Extract features using FeatureExtractor
m2a_train_features = feature_extractor_m2a.create_features(
    metabooks, amazon_books, m2a_correspondences[['id1', 'id2']], labels=m2a_correspondences['label'], id_column='id'
)

# Extract features using FeatureExtractor
m2g_train_features = feature_extractor_m2g.create_features(
    metabooks, goodreads, m2g_correspondences[['id1', 'id2']], labels=m2g_correspondences['label'], id_column='id'
)

# Prepare data for ML training
m2a_feature_columns = [col for col in m2a_train_features.columns if col not in ['id1', 'id2', 'label']]

m2a_X_train_features = m2a_train_features[m2a_feature_columns]
m2a_y_train = m2a_train_features['label']

m2g_feature_columns = [col for col in m2g_train_features.columns if col not in ['id1', 'id2', 'label']]

m2g_X_train_features = m2g_train_features[m2g_feature_columns]
m2g_y_train = m2g_train_features['label']

[INFO ] root - Label distribution: 123 positive, 476 negative
[INFO ] root - Label distribution: 132 positive, 466 negative


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import make_scorer, f1_score

# === use your real train variables here ===
X_train = m2a_X_train_features 
y_train = m2a_y_train           

param_grids = {
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42, n_jobs=-1),
        'params': {
            'n_estimators': [50, 100, 200],
            'max_depth': [5, 10, None],
            'min_samples_split': [2, 5],
            'class_weight': ['balanced', None],
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(random_state=42, max_iter=1000),  
        'params': {
            'C': [0.1, 1.0, 10.0],
            'penalty': ['l2'],
            'class_weight': ['balanced', None],
        }
    },
    'GradientBoosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [50, 100],
            'learning_rate': [0.1, 0.2],
            'max_depth': [3, 5],
        }
    },
    'SVM': {
        'model': SVC(random_state=42, probability=True), 
        'params': {
            'C': [0.1, 1.0, 10.0],
            'kernel': ['rbf', 'linear'],
            'class_weight': ['balanced', None],
 
        }
    }
}

scorer = make_scorer(f1_score)  
cv_folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(f"GridSearch setup: {len(param_grids)} models, F1 scoring, 5-fold CV")
print("\n🚀 Training Models with GridSearchCV...")

grid_search_results = {}
best_overall_score = -1
best_overall_model = None
best_model_name = None

for model_name, config in param_grids.items():
    print(f"\nTraining {model_name}...")

    grid_search = GridSearchCV(
        estimator=config['model'],
        param_grid=config['params'],
        scoring=scorer,
        cv=cv_folds,
        n_jobs=-1,
        verbose=0,
        refit=True,  
    )

    grid_search.fit(X_train, y_train)

    grid_search_results[model_name] = {
        'grid_search': grid_search,
        'best_score': grid_search.best_score_,
        'best_params': grid_search.best_params_,
        'best_estimator': grid_search.best_estimator_,
    }

    print(f"  ✅ {model_name}: Best CV F1 = {grid_search.best_score_:.4f}")
    print(f"     Best params: {grid_search.best_params_}")

    if grid_search.best_score_ > best_overall_score:
        best_overall_score = grid_search.best_score_
        best_overall_model = grid_search.best_estimator_ 
        best_model_name = model_name

print(f"\n🏆 Best Overall Model: {best_model_name} (CV F1: {best_overall_score:.4f})")
# best_overall_model now holds the fitted best estimator


GridSearch setup: 4 models, F1 scoring, 5-fold CV

🚀 Training Models with GridSearchCV...

Training RandomForest...


  ✅ RandomForest: Best CV F1 = 0.8858
     Best params: {'class_weight': 'balanced', 'max_depth': None, 'min_samples_split': 5, 'n_estimators': 50}

Training LogisticRegression...
  ✅ LogisticRegression: Best CV F1 = 0.8645
     Best params: {'C': 10.0, 'class_weight': None, 'penalty': 'l2'}

Training GradientBoosting...
  ✅ GradientBoosting: Best CV F1 = 0.8940
     Best params: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100}

Training SVM...
  ✅ SVM: Best CV F1 = 0.8643
     Best params: {'C': 10.0, 'class_weight': None, 'kernel': 'linear'}

🏆 Best Overall Model: GradientBoosting (CV F1: 0.8940)


In [ ]:
gb_best_params = grid_search_results['GradientBoosting']['best_params']
rf_best_params = grid_search_results['RandomForest']['best_params']

# --- 2) Refit models on the full training set with those exact params ---
gb_refit = GradientBoostingClassifier(random_state=42, **gb_best_params)
gb_refit.fit(X_train, y_train)

rf_refit = RandomForestClassifier(random_state=42, n_jobs=-1, **rf_best_params)
rf_refit.fit(X_train, y_train)

# --- 3) Compute feature importances ---
gb_importances = pd.Series(gb_refit.feature_importances_, index=X_train.columns, name="GB_Importance")
rf_importances = pd.Series(rf_refit.feature_importances_, index=X_train.columns, name="RF_Importance")

# Combine for easy comparison
importances = pd.concat([gb_importances, rf_importances], axis=1).fillna(0)
importances["Mean"] = importances.mean(axis=1)
importances_sorted = importances.sort_values("Mean", ascending=False)


print("\nTop features (by average importance across GB & RF):")
importances_sorted


Top features (by average importance across GB & RF):


,GB_Importance,RF_Importance,Mean
"StringComparator(title_norm, cosine, tokenization=word, list_strategy=None)",0.803292,0.285991,0.544642
"StringComparator(title_norm, jaccard, tokenization=word, list_strategy=None)",0.124123,0.255357,0.189740
"StringComparator(title_norm, jaro, tokenization=char, list_strategy=None)",0.022514,0.222085,0.122300
"StringComparator(title_norm, jaro_winkler, tokenization=char, list_strategy=None)",0.029372,0.119752,0.074562
"StringComparator(author_norm, jaro, tokenization=char, list_strategy=None)",0.004242,0.038014,0.021128
"StringComparator(author_norm, cosine, tokenization=word, list_strategy=None)",0.001067,0.039896,0.020481
"NumericComparator(publish_year, absolute_difference, list_strategy=None)",0.014404,0.017687,0.016046
"StringComparator(author_norm, jaccard, tokenization=word, list_strategy=None)",0.000986,0.021218,0.011102
